<a href="https://colab.research.google.com/github/ManPatel448/OpenCV---Based-System-for-Crack-Identification-and-Width-Estimation/blob/main/OpenCV_Based_System_For_Crack_Identification_and_Width_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SECTION 1: INSTALL & IMPORT LIBRARIES


In [ ]:
!pip -q install opencv-python scikit-image pandas matplotlib pillow

# WORKING: Import libraries
import os
import cv2
import math
import shutil
import zipfile
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import files
from skimage.morphology import skeletonize

print(f" OpenCV version: {cv2.__version__}")
print(f" NumPy version: {np.__version__}")
print(f" Pandas version: {pd.__version__}")
print(" All required libraries imported successfully.")

# SECTION 2: PROJECT CONFIGURATION

In [ ]:
PROJECT_NAME = "OpenCV-Based System for Crack Identification and Width Estimation"

# ------------------------------------------------
# Output directories
# ------------------------------------------------

OUTPUT_DIR = "crack_results"

ANNOTATED_DIR = os.path.join(
    OUTPUT_DIR,
    "annotated_images"
)

MASK_DIR = os.path.join(
    OUTPUT_DIR,
    "crack_masks"
)

PROCESSED_DIR = os.path.join(
    OUTPUT_DIR,
    "processed_images"
)

REPORT_DIR = os.path.join(
    OUTPUT_DIR,
    "reports"
)

PIPELINE_DIR = os.path.join(
    PROCESSED_DIR,
    "pipeline_stages"
)

# ------------------------------------------------
# Image processing parameters
# ------------------------------------------------

MAX_IMAGE_SIZE = 1600

GAUSSIAN_KERNEL = 5

CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID = (8, 8)

BLACKHAT_KERNEL = 21

MORPH_KERNEL_SIZE = 3
MORPH_CLOSE_ITERATIONS = 2
MORPH_OPEN_ITERATIONS = 1

MIN_CRACK_AREA = 20

MEASUREMENT_INTERVAL = 20

# ------------------------------------------------
# Calibration
# ------------------------------------------------

CALIBRATION_ENABLED = True

# Number of pixels corresponding to 1 mm
PIXELS_PER_MM = 10.0

# ------------------------------------------------
# Severity thresholds
# ------------------------------------------------

GOOD_THRESHOLD_MM = 0.30
MEDIUM_THRESHOLD_MM = 1.00

# ------------------------------------------------
# Visualization
# ------------------------------------------------

SHOW_PIPELINE = True
SHOW_INDIVIDUAL_RESULTS = True

# Maximum number of displayed measurement points
MAX_DISPLAY_POINTS = 40

print(f"Project Name       : {PROJECT_NAME}")
print(f"Maximum Image Size : {MAX_IMAGE_SIZE}")
print(f"Gaussian Kernel    : {GAUSSIAN_KERNEL}")
print(f"CLAHE Clip Limit   : {CLAHE_CLIP_LIMIT}")
print(f"BlackHat Kernel    : {BLACKHAT_KERNEL}")
print(f"Morphology Kernel  : {MORPH_KERNEL_SIZE}")
print(f"Minimum Crack Area : {MIN_CRACK_AREA}")
print(f"Measurement Step   : {MEASUREMENT_INTERVAL}")

if CALIBRATION_ENABLED:
    print(f"Calibration        : ENABLED")
    print(f"Pixels per MM      : {PIXELS_PER_MM}")
    print(f"MM per Pixel       : {1 / PIXELS_PER_MM}")
else:
    print("Calibration        : DISABLED")

print(f"GOOD threshold     : < {GOOD_THRESHOLD_MM} mm")
print(f"MEDIUM threshold   : {GOOD_THRESHOLD_MM} - {MEDIUM_THRESHOLD_MM} mm")
print(f"HEAVY threshold    : > {MEDIUM_THRESHOLD_MM} mm")

print("Project Configuration completed successfully.")
print("Output directories successfully.")
print("Image processing parameters successfully.")
print("Calibration successfully.")
print("Severity thresholds successfully.")
print("Visualization successfully.")

# SECTION 3: CREATE OUTPUT DIRECTORIES

In [ ]:
directories = [
    OUTPUT_DIR,
    ANNOTATED_DIR,
    MASK_DIR,
    PROCESSED_DIR,
    REPORT_DIR,
    PIPELINE_DIR
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"Directory ready: {directory}")

print("All output directories created successfully.")

# SECTION 4: MULTIPLE IMAGE UPLOAD

In [ ]:
ALLOWED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff"
}

print("Please select one or multiple crack images.")

uploaded = files.upload()

image_paths = []

for filename in uploaded.keys():

    extension = os.path.splitext(filename)[1].lower()

    if extension in ALLOWED_EXTENSIONS:
        image_paths.append(filename)
        print(f" Accepted image: {filename}")
    else:
        print(f" Skipped unsupported file: {filename}")

if len(image_paths) == 0:
    raise RuntimeError(
        "No supported image files were uploaded."
    )

print("---------------------------------------------------------------")
print(f"Total images uploaded: {len(image_paths)}")
print("---------------------------------------------------------------")

for i, path in enumerate(image_paths, start=1):
    print(f"{i}. {path}")

print(" Image upload successfully.")

# SECTION 5: IMAGE READING FUNCTION

In [ ]:
def read_image(image_path):
    """Read an image using OpenCV."""

    if not os.path.exists(image_path):
        raise FileNotFoundError(
            f"Image file does not exist: {image_path}"
        )

    image = cv2.imread(image_path)

    if image is None:
        raise ValueError(
            f"OpenCV could not read image: {image_path}"
        )

    if image.size == 0:
        raise ValueError(
            f"Image is empty: {image_path}"
        )

    return image


def get_image_info(image):
    """
    Return image dimensions and channel information.
    """

    height, width = image.shape[:2]

    if len(image.shape) == 2:
        channels = 1
    else:
        channels = image.shape[2]

    return {
        "width": width,
        "height": height,
        "channels": channels
    }


print("Image reading functions created successfully.")

# SECTION 6: IMAGE RESIZING

In [ ]:
def resize_image(image, max_size=MAX_IMAGE_SIZE):

    height, width = image.shape[:2]

    largest_dimension = max(height, width)

    if largest_dimension <= max_size:
        return image.copy(), 1.0

    scale = max_size / float(largest_dimension)

    new_width = max(1, int(round(width * scale)))
    new_height = max(1, int(round(height * scale)))

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )

    return resized, scale


print("Image resizing function created successfully.")

# SECTION 7: GRAYSCALE CONVERSION

In [ ]:
def convert_to_grayscale(image):

    if len(image.shape) == 2:
        return image.copy()

    return cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

print("Grayscale conversion function created successfully.")

# SECTION 8: GAUSSIAN DENOISING

In [ ]:
def apply_gaussian_blur(gray):

    kernel = int(GAUSSIAN_KERNEL)

    if kernel % 2 == 0:
        kernel += 1

    blurred = cv2.GaussianBlur(
        gray,
        (kernel, kernel),
        0
    )

    return blurred


print("Gaussian denoising function created successfully.")

# SECTION 9: CLAHE CONTRAST ENHANCEMENT

In [ ]:
def apply_clahe(gray):

    clahe = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP_LIMIT,
        tileGridSize=CLAHE_TILE_GRID
    )

    enhanced = clahe.apply(gray)

    return enhanced

print("CLAHE enhancement function created successfully.")

# SECTION 10: BLACKHAT CRACK ENHANCEMENT

In [ ]:
def apply_blackhat(image):

    kernel_size = int(BLACKHAT_KERNEL)

    if kernel_size % 2 == 0:
        kernel_size += 1

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (kernel_size, kernel_size)
    )

    blackhat = cv2.morphologyEx(
        image,
        cv2.MORPH_BLACKHAT,
        kernel
    )

    return blackhat

print("BlackHat crack enhancement function created successfully.")

# SECTION 11: OTSU THRESHOLDING

In [ ]:
def apply_otsu_threshold(blackhat):

    threshold_value, binary = cv2.threshold(
        blackhat,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    return binary, threshold_value

print("Otsu thresholding function created successfully.")

# SECTION 12: MORPHOLOGICAL NOISE REMOVAL

In [ ]:
def clean_binary_mask(binary):

    kernel_size = int(MORPH_KERNEL_SIZE)

    if kernel_size < 1:
        kernel_size = 3

    kernel = np.ones(
        (kernel_size, kernel_size),
        np.uint8
    )

    # Remove small isolated noise
    cleaned = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        kernel,
        iterations=MORPH_OPEN_ITERATIONS
    )

    # Close small gaps in crack regions
    cleaned = cv2.morphologyEx(
        cleaned,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=MORPH_CLOSE_ITERATIONS
    )

    return cleaned

print("Morphological processing function created successfully.")

# SECTION 13: CRACK SEGMENTATION

In [ ]:
def segment_cracks(clean_mask, min_area=MIN_CRACK_AREA):

    binary_uint8 = np.where(
        clean_mask > 0,
        255,
        0
    ).astype(np.uint8)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_uint8,
        connectivity=8
    )

    segmented = np.zeros_like(binary_uint8)

    valid_regions = []

    for label_id in range(1, num_labels):

        area = stats[label_id, cv2.CC_STAT_AREA]

        if area >= min_area:

            segmented[labels == label_id] = 255

            valid_regions.append({
                "label": label_id,
                "area": int(area),
                "centroid_x": float(centroids[label_id][0]),
                "centroid_y": float(centroids[label_id][1])
            })

    return segmented, valid_regions

print("Crack segmentation function created successfully.")

# SECTION 14: CONTOUR DETECTION

In [ ]:
def detect_crack_contours(mask, min_area=MIN_CRACK_AREA):

    contours, hierarchy = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    valid_contours = []

    for contour in contours:

        area = cv2.contourArea(contour)

        if area >= min_area:
            valid_contours.append(contour)

    valid_contours.sort(
        key=cv2.contourArea,
        reverse=True
    )

    return valid_contours


def contour_information(contours):

    information = []

    for crack_id, contour in enumerate(contours, start=1):

        area = float(cv2.contourArea(contour))
        perimeter = float(cv2.arcLength(contour, True))

        x, y, w, h = cv2.boundingRect(contour)

        moments = cv2.moments(contour)

        if moments["m00"] != 0:
            cx = moments["m10"] / moments["m00"]
            cy = moments["m01"] / moments["m00"]
        else:
            cx = x + w / 2
            cy = y + h / 2

        information.append({
            "crack_id": crack_id,
            "area_pixels": area,
            "perimeter_pixels": perimeter,
            "bbox_x": x,
            "bbox_y": y,
            "bbox_width": w,
            "bbox_height": h,
            "centroid_x": cx,
            "centroid_y": cy
        })

    return information


print("Contour detection functions created successfully.")

# SECTION 15: CRACK AREA ESTIMATION

In [ ]:
def calculate_crack_area(contours):

    total_area_pixels = 0.0

    areas = []

    for crack_id, contour in enumerate(contours, start=1):

        area_pixels = float(
            cv2.contourArea(contour)
        )

        total_area_pixels += area_pixels

        if CALIBRATION_ENABLED:
            area_mm2 = (
                area_pixels /
                (PIXELS_PER_MM ** 2)
            )
        else:
            area_mm2 = np.nan

        areas.append({
            "crack_id": crack_id,
            "area_pixels": area_pixels,
            "area_mm2": area_mm2
        })

    if CALIBRATION_ENABLED:
        total_area_mm2 = (
            total_area_pixels /
            (PIXELS_PER_MM ** 2)
        )
    else:
        total_area_mm2 = np.nan

    return (
        total_area_pixels,
        total_area_mm2,
        areas
    )

print("Crack area calculation function created successfully.")

# SECTION 16: CRACK SKELETONIZATION

In [ ]:
def skeletonize_crack(mask):

    binary_bool = mask > 0

    skeleton_bool = skeletonize(
        binary_bool
    )

    skeleton_uint8 = (
        skeleton_bool.astype(np.uint8) * 255
    )

    return skeleton_bool, skeleton_uint8

print("Skeletonization function created successfully.")

# SECTION 17: CRACK LENGTH ESTIMATION

In [ ]:
def calculate_skeleton_length(skeleton):

    # A simple and robust approximation for skeleton length is the count of its pixels.
    # The skeleton parameter is already a boolean numpy array from skeletonize_crack.
    return float(np.count_nonzero(skeleton))


def convert_length_to_mm(length_pixels):

    if not CALIBRATION_ENABLED:
        return np.nan

    return (
        length_pixels /
        PIXELS_PER_MM
    )

# SECTION 18: DISTANCE TRANSFORM

In [ ]:
def calculate_distance_transform(mask):

    binary_mask = np.where(
        mask > 0,
        255,
        0
    ).astype(np.uint8)

    distance_map = cv2.distanceTransform(
        binary_mask,
        cv2.DIST_L2,
        5
    )

    return distance_map


def create_distance_visualization(distance_map):

    if distance_map is None:
        return None

    max_value = float(np.max(distance_map))

    if max_value <= 0:
        normalized = np.zeros_like(
            distance_map,
            dtype=np.uint8
        )
    else:
        normalized = cv2.normalize(
            distance_map,
            None,
            0,
            255,
            cv2.NORM_MINMAX
        ).astype(np.uint8)

    colored = cv2.applyColorMap(
        normalized,
        cv2.COLORMAP_JET
    )

    return colored


print("Distance transform functions created successfully.")

# SECTION 19: CRACK WIDTH ESTIMATION

In [ ]:
# WORKING:
# Width = 2 × Distance Transform Value
# Distance Transform Value represents the radius from a skeleton
# pixel to the nearest crack boundary.

def estimate_crack_width(
    skeleton,
    distance_map,
    pixels_per_mm=PIXELS_PER_MM
):

    rows, cols = np.where(
        skeleton.astype(bool)
    )

    measurements = []

    for point_id, (y, x) in enumerate(
        zip(rows, cols),
        start=1
    ):

        distance_value = float(
            distance_map[y, x]
        )

        if distance_value <= 0:
            continue

        # CORE RESEARCH FORMULA
        width_pixels = (
            2.0 * distance_value
        )

        if CALIBRATION_ENABLED:
            width_mm = (
                width_pixels /
                pixels_per_mm
            )
        else:
            width_mm = np.nan

        measurements.append({
            "point_id": point_id,
            "x": int(x),
            "y": int(y),
            "distance_pixels": distance_value,
            "width_pixels": width_pixels,
            "width_mm": width_mm
        })

    if len(measurements) == 0:

        return (
            pd.DataFrame(),
            {
                "minimum_width_pixels": np.nan,
                "maximum_width_pixels": np.nan,
                "average_width_pixels": np.nan,
                "median_width_pixels": np.nan,
                "minimum_width_mm": np.nan,
                "maximum_width_mm": np.nan,
                "average_width_mm": np.nan,
                "median_width_mm": np.nan
            }
        )

    measurement_df = pd.DataFrame(
        measurements
    )

    width_pixels_values = (
        measurement_df["width_pixels"]
        .to_numpy()
    )

    result = {
        "minimum_width_pixels": float(
            np.min(width_pixels_values)
        ),
        "maximum_width_pixels": float(
            np.max(width_pixels_values)
        ),
        "average_width_pixels": float(
            np.mean(width_pixels_values)
        ),
        "median_width_pixels": float(
            np.median(width_pixels_values)
        )
    }

    if CALIBRATION_ENABLED:

        width_mm_values = (
            measurement_df["width_mm"]
            .to_numpy()
        )

        result.update({
            "minimum_width_mm": float(
                np.min(width_mm_values)
            ),
            "maximum_width_mm": float(
                np.max(width_mm_values)
            ),
            "average_width_mm": float(
                np.mean(width_mm_values)
            ),
            "median_width_mm": float(
                np.median(width_mm_values)
            )
        })

    else:

        result.update({
            "minimum_width_mm": np.nan,
            "maximum_width_mm": np.nan,
            "average_width_mm": np.nan,
            "median_width_mm": np.nan
        })

    return measurement_df, result


print("Crack width estimation function created successfully.")
print("Formula: Width = 2 × Distance Transform Value")

# SECTION 20: MEASUREMENT POINT SELECTION

In [ ]:
# WORKING: Select readable measurement points instead of annotating
# every skeleton pixel.

def select_measurement_points(
    measurement_df,
    interval=MEASUREMENT_INTERVAL,
    max_points=MAX_DISPLAY_POINTS
):

    if measurement_df.empty:
        return measurement_df.copy()

    interval = max(1, int(interval))

    selected = measurement_df.iloc[
        ::interval
    ].copy()

    # Always include maximum-width point
    max_index = measurement_df[
        "width_pixels"
    ].idxmax()

    max_row = measurement_df.loc[
        [max_index]
    ]

    selected = pd.concat(
        [selected, max_row],
        ignore_index=False
    )

    # Remove duplicate point IDs
    selected = selected.drop_duplicates(
        subset=["point_id"]
    )

    # Limit display count while preserving max-width point
    if len(selected) > max_points:

        selected = selected.iloc[
            :max_points
        ].copy()

        if max_index not in selected.index:

            selected = pd.concat(
                [
                    selected.iloc[:-1],
                    max_row
                ]
            )

    return selected.reset_index(
        drop=True
    )


print("Measurement point selection function created successfully.")

# SECTION 21: CRACK SEVERITY CLASSIFICATION

In [ ]:
# WORKING: Classify crack severity based on maximum estimated width

def classify_severity(max_width_mm):

    if not CALIBRATION_ENABLED:
        return (
            "N/A",
            "N/A"
        )

    if not np.isfinite(max_width_mm):
        return (
            "N/A",
            "N/A"
        )

    if max_width_mm < GOOD_THRESHOLD_MM:

        return (
            "GOOD",
            "LOW"
        )

    elif max_width_mm <= MEDIUM_THRESHOLD_MM:

        return (
            "MEDIUM",
            "MEDIUM"
        )

    else:

        return (
            "HEAVY",
            "HIGH"
        )


def generate_health_message(
    severity,
    risk_level,
    crack_detected=True
):

    if not crack_detected:

        return (
            "No significant crack was detected using the current "
            "image-processing parameters."
        )

    if severity == "GOOD":

        return (
            "Crack width is below 0.30 mm. The detected crack is "
            "classified as low risk. Periodic monitoring is recommended."
        )

    if severity == "MEDIUM":

        return (
            "Crack width is between 0.30 mm and 1.00 mm. The detected "
            "crack is classified as medium risk. Inspection and repair "
            "are recommended."
        )

    if severity == "HEAVY":

        return (
            "Crack width is greater than 1.00 mm. The detected crack is "
            "classified as high risk. Detailed inspection and appropriate "
            "repair or strengthening should be considered."
        )

    return (
        "Calibration is required for millimeter-based severity classification."
    )


print(" Severity classification functions created successfully.")

# SECTION 22: WIDTH MEASUREMENT ANNOTATION

In [ ]:
# WORKING: Draw skeleton, measurement points and approximate
# perpendicular crack-width lines.

def get_local_tangent(
    skeleton,
    x,
    y,
    radius=5
):

    height, width = skeleton.shape

    points = []

    for yy in range(
        max(0, y - radius),
        min(height, y + radius + 1)
    ):

        for xx in range(
            max(0, x - radius),
            min(width, x + radius + 1)
        ):

            if xx == x and yy == y:
                continue

            if skeleton[yy, xx]:
                points.append(
                    (xx, yy)
                )

    if len(points) < 2:
        return np.array([1.0, 0.0])

    coords = np.array(
        points,
        dtype=np.float32
    )

    center = np.array(
        [x, y],
        dtype=np.float32
    )

    vectors = coords - center

    # PCA gives dominant local direction
    covariance = np.cov(
        vectors.T
    )

    if covariance.ndim != 2:
        return np.array([1.0, 0.0])

    eigenvalues, eigenvectors = np.linalg.eigh(
        covariance
    )

    tangent = eigenvectors[
        :, np.argmax(eigenvalues)
    ]

    norm = np.linalg.norm(tangent)

    if norm == 0:
        return np.array([1.0, 0.0])

    return tangent / norm


def draw_width_line(
    image,
    x,
    y,
    width_pixels,
    skeleton,
    line_length_factor=0.5
):

    tangent = get_local_tangent(
        skeleton,
        x,
        y
    )

    # Perpendicular direction
    normal = np.array(
        [-tangent[1], tangent[0]]
    )

    normal_norm = np.linalg.norm(
        normal
    )

    if normal_norm == 0:
        normal = np.array([0.0, 1.0])
    else:
        normal = normal / normal_norm

    half_width = (
        width_pixels *
        line_length_factor
    )

    x1 = int(
        round(
            x - normal[0] * half_width
        )
    )

    y1 = int(
        round(
            y - normal[1] * half_width
        )
    )

    x2 = int(
        round(
            x + normal[0] * half_width
        )
    )

    y2 = int(
        round(
            y + normal[1] * half_width
        )
    )

    cv2.line(
        image,
        (x1, y1),
        (x2, y2),
        (255, 255, 0),
        2
    )

    cv2.circle(
        image,
        (x, y),
        4,
        (255, 0, 255),
        -1
    )

    return image


def severity_display_color(severity):

    if severity == "GOOD":
        return (0, 180, 0)

    if severity == "MEDIUM":
        return (0, 165, 255)

    if severity == "HEAVY":
        return (0, 0, 255)

    return (255, 255, 255)


def create_width_annotations(
    image,
    skeleton,
    selected_points,
    severity
):

    annotated = image.copy()

    color = severity_display_color(
        severity
    )

    # Draw skeleton in green
    skeleton_overlay = np.zeros_like(
        annotated
    )

    skeleton_overlay[
        skeleton.astype(bool)
    ] = (0, 255, 0)

    annotated = cv2.addWeighted(
        annotated,
        1.0,
        skeleton_overlay,
        0.7,
        0
    )

    for _, row in selected_points.iterrows():

        x = int(row["x"])
        y = int(row["y"])

        width_pixels = float(
            row["width_pixels"]
        )

        if CALIBRATION_ENABLED:

            label = (
                f'{row["width_mm"]:.2f} mm'
            )

        else:

            label = (
                f'{width_pixels:.2f} px'
            )

        draw_width_line(
            annotated,
            x,
            y,
            width_pixels,
            skeleton
        )

        cv2.circle(
            annotated,
            (x, y),
            5,
            color,
            -1
        )

        text_x = min(
            max(5, x + 8),
            max(5, annotated.shape[1] - 150)
        )

        text_y = min(
            max(20, y - 8),
            max(20, annotated.shape[0] - 10)
        )

        # Black background rectangle for readability
        (tw, th), baseline = cv2.getTextSize(
            label,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            2
        )

        cv2.rectangle(
            annotated,
            (
                text_x - 3,
                text_y - th - 5
            ),
            (
                text_x + tw + 3,
                text_y + baseline + 3
            ),
            (0, 0, 0),
            -1
        )

        cv2.putText(
            annotated,
            label,
            (text_x, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            color,
            2,
            cv2.LINE_AA
        )

    return annotated


print("Width annotation functions created successfully.")

# SECTION 23: FINAL RESULT ANNOTATION

In [ ]:
# WORKING: Create final professional crack analysis image

def create_final_annotation(
    original_image,
    contours,
    skeleton,
    selected_points,
    result
):

    annotated = original_image.copy()

    severity = result.get(
        "severity",
        "N/A"
    )

    risk_level = result.get(
        "risk_level",
        "N/A"
    )

    # ------------------------------------------------
    # Draw crack contours
    # ------------------------------------------------

    if len(contours) > 0:

        cv2.drawContours(
            annotated,
            contours,
            -1,
            (255, 0, 255),
            2
        )

    # ------------------------------------------------
    # Draw skeleton
    # ------------------------------------------------

    skeleton_mask = skeleton.astype(bool)

    annotated[
        skeleton_mask
    ] = (0, 255, 0)

    # ------------------------------------------------
    # Draw measurement points
    # ------------------------------------------------

    point_color = severity_display_color(
        severity
    )

    for _, row in selected_points.iterrows():

        x = int(row["x"])
        y = int(row["y"])

        width_pixels = float(
            row["width_pixels"]
        )

        # Width line
        annotated = draw_width_line(
            annotated,
            x,
            y,
            width_pixels,
            skeleton
        )

        # Severity point
        cv2.circle(
            annotated,
            (x, y),
            5,
            point_color,
            -1
        )

        # Width text
        if CALIBRATION_ENABLED:

            text = (
                f'{row["width_mm"]:.2f} mm'
            )

        else:

            text = (
                f'{width_pixels:.2f} px'
            )

        tx = x + 10
        ty = y - 10

        tx = max(
            5,
            min(
                tx,
                annotated.shape[1] - 130
            )
        )

        ty = max(
            20,
            min(
                ty,
                annotated.shape[0] - 10
            )
        )

        cv2.putText(
            annotated,
            text,
            (tx, ty),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            point_color,
            2,
            cv2.LINE_AA
        )

    # ------------------------------------------------
    # Result information panel
    # ------------------------------------------------

    panel_width = min(
        470,
        max(300, annotated.shape[1] // 3)
    )

    panel_height = 260

    overlay = annotated.copy()

    cv2.rectangle(
        overlay,
        (10, 10),
        (
            panel_width,
            panel_height
        ),
        (0, 0, 0),
        -1
    )

    annotated = cv2.addWeighted(
        overlay,
        0.72,
        annotated,
        0.28,
        0
    )

    cv2.putText(
        annotated,
        "CRACK ANALYSIS RESULT",
        (25, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    y_position = 72
    line_gap = 28

    if result.get("crack_detected", False):

        if CALIBRATION_ENABLED:

            max_width_text = (
                f'{result["maximum_width_mm"]:.3f} mm'
            )

            avg_width_text = (
                f'{result["average_width_mm"]:.3f} mm'
            )

            length_text = (
                f'{result["crack_length_mm"]:.3f} mm'
            )

            area_text = (
                f'{result["crack_area_mm2"]:.3f} mm2'
            )

        else:

            max_width_text = (
                f'{result["maximum_width_pixels"]:.3f} px'
            )

            avg_width_text = (
                f'{result["average_width_pixels"]:.3f} px'
            )

            length_text = (
                f'{result["crack_length_pixels"]:.3f} px'
            )

            area_text = (
                f'{result["crack_area_pixels"]:.3f} px2'
            )

        information = [
            f'Crack Detected : YES',
            f'Crack Regions  : {result["crack_regions"]}',
            f'Max Width      : {max_width_text}',
            f'Average Width  : {avg_width_text}',
            f'Crack Length   : {length_text}',
            f'Crack Area     : {area_text}',
            f'Severity       : {severity}',
            f'Risk Level     : {risk_level}'
        ]

    else:

        information = [
            'Crack Detected : NO',
            'Crack Regions  : 0',
            'Severity       : N/A',
            'Risk Level     : N/A'
        ]

    for line in information:

        cv2.putText(
            annotated,
            line,
            (25, y_position),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.52,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )

        y_position += line_gap

    return annotated


print("Final annotation function created successfully.")

# SECTION 24: PIPELINE VISUALIZATION

In [ ]:
# WORKING: Display all OpenCV processing stages
def display_pipeline(stages, filename):

    stage_items = [
        ("1. Original Image", stages.get("original")),
        ("2. Resized Image", stages.get("resized")),
        ("3. Grayscale", stages.get("gray")),
        ("4. Gaussian Blur", stages.get("blurred")),
        ("5. CLAHE", stages.get("clahe")),
        ("6. BlackHat", stages.get("blackhat")),
        ("7. Otsu Threshold", stages.get("binary")),
        ("8. Morphological Mask", stages.get("morphology")),
        ("9. Crack Segmentation", stages.get("segmented")),
        ("10. Contours", stages.get("contours")),
        ("11. Skeleton", stages.get("skeleton")),
        ("12. Distance Transform", stages.get("distance_visual")),
        ("13. Width Measurement", stages.get("width")),
        ("14. Final Result", stages.get("final"))
    ]

    valid_items = [
        item
        for item in stage_items
        if item[1] is not None
    ]

    if not valid_items:
        return

    for title, image in valid_items:

        plt.figure(figsize=(11, 6))

        if len(image.shape) == 2:

            plt.imshow(
                image,
                cmap="gray"
            )

        else:

            image_rgb = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            )

            plt.imshow(
                image_rgb
            )

        plt.title(
            f"{filename} - {title}",
            fontsize=14
        )

        plt.axis("off")
        plt.show()


print("Pipeline visualization function created successfully.")

# SECTION 25: PROCESS SINGLE IMAGE

In [ ]:
# WORKING: Complete image processing pipeline
def process_single_image(image_path):

    filename = os.path.basename(
        image_path
    )

    base_name = os.path.splitext(
        filename
    )[0]

    print()
    print("---------------------------------------------------------------")
    print(f"PROCESSING: {filename}")
    print("---------------------------------------------------------------")

    stages = {}

    # ------------------------------------------------
    # 1. Read
    # ------------------------------------------------

    image = read_image(
        image_path
    )

    stages["original"] = image.copy()

    info_original = get_image_info(
        image
    )

    print(
        f" Image read: "
        f"{info_original['width']} x "
        f"{info_original['height']}"
    )

    # ------------------------------------------------
    # 2. Resize
    # ------------------------------------------------

    resized, resize_scale = resize_image(
        image
    )

    stages["resized"] = resized.copy()

    info_resized = get_image_info(
        resized
    )

    print(
        f" Resize: "
        f"{info_resized['width']} x "
        f"{info_resized['height']}"
    )

    # ------------------------------------------------
    # 3. Grayscale
    # ------------------------------------------------

    gray = convert_to_grayscale(
        resized
    )

    stages["gray"] = gray.copy()

    # ------------------------------------------------
    # 4. Gaussian Blur
    # ------------------------------------------------

    blurred = apply_gaussian_blur(
        gray
    )

    stages["blurred"] = blurred.copy()

    # ------------------------------------------------
    # 5. CLAHE
    # ------------------------------------------------

    clahe_image = apply_clahe(
        blurred
    )

    stages["clahe"] = clahe_image.copy()

    # ------------------------------------------------
    # 6. BlackHat
    # ------------------------------------------------

    blackhat = apply_blackhat(
        clahe_image
    )

    stages["blackhat"] = blackhat.copy()

    # ------------------------------------------------
    # 7. Otsu Threshold
    # ------------------------------------------------

    binary, otsu_value = apply_otsu_threshold(
        blackhat
    )

    stages["binary"] = binary.copy()

    print(
        f" Otsu threshold value: "
        f"{otsu_value:.2f}"
    )

    # ------------------------------------------------
    # 8. Morphology
    # ------------------------------------------------

    morphology = clean_binary_mask(
        binary
    )

    stages["morphology"] = morphology.copy()

    # ------------------------------------------------
    # 9. Segmentation
    # ------------------------------------------------

    segmented, valid_regions = segment_cracks(
        morphology
    )

    stages["segmented"] = segmented.copy()

    # ------------------------------------------------
    # 10. Contours
    # ------------------------------------------------

    contours = detect_crack_contours(
        segmented
    )

    contour_image = resized.copy()

    if len(contours) > 0:

        cv2.drawContours(
            contour_image,
            contours,
            -1,
            (0, 255, 0),
            2
        )

    stages["contours"] = contour_image

    crack_detected = len(contours) > 0

    print(
        f" Crack regions detected: "
        f"{len(contours)}"
    )

    # ------------------------------------------------
    # 11. Area
    # ------------------------------------------------

    (
        crack_area_pixels,
        crack_area_mm2,
        area_details
    ) = calculate_crack_area(
        contours
    )

    # ------------------------------------------------
    # 12. Skeleton
    # ------------------------------------------------

    skeleton_bool, skeleton_uint8 = skeletonize_crack(
        segmented
    )

    stages["skeleton"] = skeleton_uint8.copy()

    skeleton_pixel_count = int(
        np.count_nonzero(
            skeleton_bool
        )
    )

    print(
        f" Skeleton pixels: "
        f"{skeleton_pixel_count}"
    )

    # ------------------------------------------------
    # 13. Length
    # ------------------------------------------------

    crack_length_pixels = calculate_skeleton_length(
        skeleton_bool
    )

    crack_length_mm = convert_length_to_mm(
        crack_length_pixels
    )

    # ------------------------------------------------
    # 14. Distance Transform
    # ------------------------------------------------

    distance_map = calculate_distance_transform(
        segmented
    )

    distance_visual = create_distance_visualization(
        distance_map
    )

    stages["distance_visual"] = distance_visual

    # ------------------------------------------------
    # 15. Width
    # ------------------------------------------------

    measurement_df, width_stats = estimate_crack_width(
        skeleton_bool,
        distance_map
    )

    if not measurement_df.empty:

        selected_points = select_measurement_points(
            measurement_df
        )

    else:

        selected_points = pd.DataFrame()

    # ------------------------------------------------
    # 16. Severity
    # ------------------------------------------------

    if crack_detected and not measurement_df.empty:

        if CALIBRATION_ENABLED:

            maximum_width_mm = (
                width_stats["maximum_width_mm"]
            )

            severity, risk_level = classify_severity(
                maximum_width_mm
            )

        else:

            maximum_width_mm = np.nan

            severity = "N/A"
            risk_level = "N/A"

        message = generate_health_message(
            severity,
            risk_level,
            crack_detected=True
        )

    else:

        severity = "N/A"
        risk_level = "N/A"

        message = generate_health_message(
            severity,
            risk_level,
            crack_detected=False
        )

    # ------------------------------------------------
    # 17. Create width annotation
    # ------------------------------------------------

    if not selected_points.empty:

        width_annotation = create_width_annotations(
            resized,
            skeleton_bool,
            selected_points,
            severity
        )

    else:

        width_annotation = resized.copy()

    stages["width"] = width_annotation.copy()

    # ------------------------------------------------
    # 18. Final annotation
    # ------------------------------------------------

    preliminary_result = {
        "crack_detected": crack_detected,
        "crack_regions": len(contours),
        "crack_area_pixels": crack_area_pixels,
        "crack_area_mm2": crack_area_mm2,
        "crack_length_pixels": crack_length_pixels,
        "crack_length_mm": crack_length_mm,
        "maximum_width_pixels": width_stats[
            "maximum_width_pixels"
        ],
        "average_width_pixels": width_stats[
            "average_width_pixels"
        ],
        "minimum_width_pixels": width_stats[
            "minimum_width_pixels"
        ],
        "median_width_pixels": width_stats[
            "median_width_pixels"
        ],
        "minimum_width_mm": width_stats[
            "minimum_width_mm"
        ],
        "maximum_width_mm": width_stats[
            "maximum_width_mm"
        ],
        "average_width_mm": width_stats[
            "average_width_mm"
        ],
        "median_width_mm": width_stats[
            "median_width_mm"
        ],
        "severity": severity,
        "risk_level": risk_level,
        "message": message
    }

    final_annotation = create_final_annotation(
        resized,
        contours,
        skeleton_bool,
        selected_points,
        preliminary_result
    )

    stages["final"] = final_annotation.copy()

    # ------------------------------------------------
    # 19. Save output images
    # ------------------------------------------------

    mask_path = os.path.join(
        MASK_DIR,
        f"{base_name}_mask.png"
    )

    skeleton_path = os.path.join(
        PROCESSED_DIR,
        f"{base_name}_skeleton.png"
    )

    distance_path = os.path.join(
        PROCESSED_DIR,
        f"{base_name}_distance_transform.png"
    )

    contour_path = os.path.join(
        PROCESSED_DIR,
        f"{base_name}_contours.png"
    )

    final_path = os.path.join(
        ANNOTATED_DIR,
        f"{base_name}_annotated.png"
    )

    cv2.imwrite(
        mask_path,
        segmented
    )

    cv2.imwrite(
        skeleton_path,
        skeleton_uint8
    )

    if distance_visual is not None:
        cv2.imwrite(
            distance_path,
            distance_visual
        )

    cv2.imwrite(
        contour_path,
        contour_image
    )

    cv2.imwrite(
        final_path,
        final_annotation
    )

    # ------------------------------------------------
    # 20. Save individual pipeline stages
    # ------------------------------------------------

    image_pipeline_dir = os.path.join(
        PIPELINE_DIR,
        base_name
    )

    os.makedirs(
        image_pipeline_dir,
        exist_ok=True
    )

    pipeline_save_map = {
        "01_original.png": stages["original"],
        "02_resized.png": stages["resized"],
        "03_grayscale.png": stages["gray"],
        "04_gaussian_blur.png": stages["blurred"],
        "05_clahe.png": stages["clahe"],
        "06_blackhat.png": stages["blackhat"],
        "07_otsu_binary.png": stages["binary"],
        "08_morphology.png": stages["morphology"],
        "09_segmentation.png": stages["segmented"],
        "10_contours.png": stages["contours"],
        "11_skeleton.png": stages["skeleton"],
        "12_distance_transform.png": stages["distance_visual"],
        "13_width_measurement.png": stages["width"],
        "14_final_result.png": stages["final"]
    }

    for stage_filename, stage_image in pipeline_save_map.items():

        if stage_image is not None:

            cv2.imwrite(
                os.path.join(
                    image_pipeline_dir,
                    stage_filename
                ),
                stage_image
            )

    # ------------------------------------------------
    # 21. Add crack IDs to measurements
    # ------------------------------------------------

    if not measurement_df.empty:

        measurement_df = measurement_df.copy()

        measurement_df.insert(
            0,
            "Image_Name",
            filename
        )

        measurement_df.insert(
            1,
            "Crack_ID",
            1
        )

        measurement_df["Severity"] = severity
        measurement_df["Risk_Level"] = risk_level

    # ------------------------------------------------
    # 22. Final result dictionary
    # ------------------------------------------------

    result = {
        "Image_Name": filename,
        "Image_Width": info_resized["width"],
        "Image_Height": info_resized["height"],
        "Original_Width": info_original["width"],
        "Original_Height": info_original["height"],
        "Resize_Scale": resize_scale,
        "Crack_Detected": "YES" if crack_detected else "NO",
        "Crack_Regions": len(contours),
        "Crack_Area_Pixels": crack_area_pixels,
        "Crack_Area_MM2": crack_area_mm2,
        "Crack_Length_Pixels": crack_length_pixels,
        "Crack_Length_MM": crack_length_mm,
        "Minimum_Width_Pixels": width_stats[
            "minimum_width_pixels"
        ],
        "Maximum_Width_Pixels": width_stats[
            "maximum_width_pixels"
        ],
        "Average_Width_Pixels": width_stats[
            "average_width_pixels"
        ],
        "Median_Width_Pixels": width_stats[
            "median_width_pixels"
        ],
        "Minimum_Width_MM": width_stats[
            "minimum_width_mm"
        ],
        "Maximum_Width_MM": width_stats[
            "maximum_width_mm"
        ],
        "Average_Width_MM": width_stats[
            "average_width_mm"
        ],
        "Median_Width_MM": width_stats[
            "median_width_mm"
        ],
        "Calibration_Enabled": CALIBRATION_ENABLED,
        "Pixels_Per_MM": PIXELS_PER_MM
            if CALIBRATION_ENABLED else np.nan,
        "Severity": severity,
        "Risk_Level": risk_level,
        "Health_Status": (
            severity
            if crack_detected
            else "NO CRACK DETECTED"
        ),
        "Automatic_Message": message,
        "Otsu_Threshold": otsu_value,
        "Skeleton_Pixels": skeleton_pixel_count,
        "Measurement_Points": (
            len(measurement_df)
            if not measurement_df.empty
            else 0
        ),
        "Mask_Path": mask_path,
        "Skeleton_Path": skeleton_path,
        "Distance_Path": distance_path,
        "Contour_Path": contour_path,
        "Annotated_Path": final_path,
        "Pipeline_Directory": image_pipeline_dir
    }

    print(
        f" Area: "
        f"{crack_area_pixels:.3f} pixels²"
    )

    print(
        f" Length: "
        f"{crack_length_pixels:.3f} pixels"
    )

    if CALIBRATION_ENABLED:

        print(
            f" Maximum Width: "
            f"{width_stats['maximum_width_mm']:.3f} mm"
        )

        print(
            f" Average Width: "
            f"{width_stats['average_width_mm']:.3f} mm"
        )

    else:

        print(
            f" Maximum Width: "
            f"{width_stats['maximum_width_pixels']:.3f} pixels"
        )

    print(
        f" Severity: {severity}"
    )

    print(
        f" Risk Level: {risk_level}"
    )

    print(
        f" Health Message: {message}"
    )

    print(
        f" {filename} processed successfully."
    )

    return (
        result,
        measurement_df,
        stages
    )


print(" Complete single-image pipeline created successfully.")

# SECTION 26: PROCESS ALL UPLOADED IMAGES

In [ ]:
# WORKING: Process every uploaded image independently
def process_all_images(image_paths):

    all_results = []
    all_measurements = []
    all_stages = {}

    successful_files = []
    failed_files = []

    total = len(image_paths)

    for index, image_path in enumerate(
        image_paths,
        start=1
    ):

        filename = os.path.basename(
            image_path
        )

        print()
        print("===============================================================")
        print(
            f"PROCESSING IMAGE {index}/{total}: {filename}"
        )
        print("===============================================================")

        try:

            result, measurement_df, stages = process_single_image(
                image_path
            )

            all_results.append(
                result
            )

            if not measurement_df.empty:

                all_measurements.append(
                    measurement_df
                )

            all_stages[
                filename
            ] = stages

            successful_files.append(
                filename
            )

        except Exception as e:

            failed_files.append(
                {
                    "filename": filename,
                    "error": str(e)
                }
            )

            print(
                f"Failed to process {filename}: {e}"
            )

            traceback.print_exc()

    return (
        all_results,
        all_measurements,
        all_stages,
        successful_files,
        failed_files
    )


(
    results,
    measurement_dataframes,
    all_stages,
    successful_files,
    failed_files
) = process_all_images(
    image_paths
)

print(
    f" Uploaded       : {len(image_paths)}"
)

print(
    f" Successful     : {len(successful_files)}"
)

print(
    f" Failed         : {len(failed_files)}"
)

print("completed successfully.")

# SECTION 27: RESEARCH SUMMARY DATAFRAME

In [ ]:
# WORKING: Convert results into a research-friendly DataFrame

if len(results) > 0:

    summary_df = pd.DataFrame(
        results
    )

else:

    summary_df = pd.DataFrame()

# Display only useful research columns first

summary_columns = [
    "Image_Name",
    "Crack_Detected",
    "Crack_Regions",
    "Crack_Area_Pixels",
    "Crack_Area_MM2",
    "Crack_Length_Pixels",
    "Crack_Length_MM",
    "Minimum_Width_Pixels",
    "Maximum_Width_Pixels",
    "Average_Width_Pixels",
    "Median_Width_Pixels",
    "Minimum_Width_MM",
    "Maximum_Width_MM",
    "Average_Width_MM",
    "Median_Width_MM",
    "Calibration_Enabled",
    "Pixels_Per_MM",
    "Severity",
    "Risk_Level",
    "Health_Status"
]

available_summary_columns = [
    column
    for column in summary_columns
    if column in summary_df.columns
]

if not summary_df.empty:

    display(
        summary_df[
            available_summary_columns
        ].round(4)
    )

else:

    print("No successful image results available.")

print("completed successfully.")

# SECTION 28: DETAILED MEASUREMENT CSV DATA

In [ ]:
# WORKING: Combine all individual skeleton measurement points

if len(measurement_dataframes) > 0:

    detailed_measurements_df = pd.concat(
        measurement_dataframes,
        ignore_index=True
    )

else:

    detailed_measurements_df = pd.DataFrame(
        columns=[
            "Image_Name",
            "Crack_ID",
            "point_id",
            "x",
            "y",
            "distance_pixels",
            "width_pixels",
            "width_mm",
            "Severity",
            "Risk_Level"
        ]
    )

# Rename columns for research-friendly CSV

detailed_measurements_df = detailed_measurements_df.rename(
    columns={
        "point_id": "Point_ID",
        "x": "X",
        "y": "Y",
        "distance_pixels": "Distance_Pixels",
        "width_pixels": "Width_Pixels",
        "width_mm": "Width_MM"
    }
)

measurement_csv_path = os.path.join(
    REPORT_DIR,
    "crack_measurement_points.csv"
)

detailed_measurements_df.to_csv(
    measurement_csv_path,
    index=False
)

print(
    f"Measurement rows: "
    f"{len(detailed_measurements_df)}"
)

print(
    f"Saved: {measurement_csv_path}"
)

if not detailed_measurements_df.empty:

    display(
        detailed_measurements_df.head(20).round(4)
    )

print("completed successfully.")

# SECTION 29: FINAL RESEARCH CSV REPORT

In [ ]:
# WORKING: Generate one summary row for every processed image
research_csv_path = os.path.join(
    REPORT_DIR,
    "crack_analysis_report.csv"
)

if not summary_df.empty:

    report_columns = [
        "Image_Name",
        "Image_Width",
        "Image_Height",
        "Original_Width",
        "Original_Height",
        "Resize_Scale",
        "Crack_Detected",
        "Crack_Regions",
        "Crack_Area_Pixels",
        "Crack_Area_MM2",
        "Crack_Length_Pixels",
        "Crack_Length_MM",
        "Minimum_Width_Pixels",
        "Maximum_Width_Pixels",
        "Average_Width_Pixels",
        "Median_Width_Pixels",
        "Minimum_Width_MM",
        "Maximum_Width_MM",
        "Average_Width_MM",
        "Median_Width_MM",
        "Calibration_Enabled",
        "Pixels_Per_MM",
        "Severity",
        "Risk_Level",
        "Health_Status",
        "Automatic_Message",
        "Otsu_Threshold",
        "Skeleton_Pixels",
        "Measurement_Points"
    ]

    report_columns = [
        column
        for column in report_columns
        if column in summary_df.columns
    ]

    final_report_df = summary_df[
        report_columns
    ].copy()

else:

    final_report_df = pd.DataFrame()

final_report_df.to_csv(
    research_csv_path,
    index=False
)

print(
    f"Report saved: {research_csv_path}"
)

if not final_report_df.empty:

    display(
        final_report_df.round(4)
    )

print("Section 29 completed successfully.")

# SECTION 30: DISPLAY FINAL RESULTS

In [ ]:
# WORKING: Display final annotated result for each successful image

if len(results) == 0:

    print("[!] No successful results to display.")

else:

    for result in results:

        image_name = result[
            "Image_Name"
        ]

        annotated_path = result[
            "Annotated_Path"
        ]

        if not os.path.exists(
            annotated_path
        ):
            continue

        annotated = cv2.imread(
            annotated_path
        )

        if annotated is None:
            continue

        annotated_rgb = cv2.cvtColor(
            annotated,
            cv2.COLOR_BGR2RGB
        )

        plt.figure(
            figsize=(14, 8)
        )

        plt.imshow(
            annotated_rgb
        )

        plt.title(
            f"{image_name} | "
            f"Severity: {result['Severity']} | "
            f"Risk: {result['Risk_Level']}",
            fontsize=16
        )

        plt.axis("off")
        plt.show()

        print("---------------------------------------------------------------")
        print(f"IMAGE: {image_name}")
        print("---------------------------------------------------------------")
        print(
            f"Crack Detected : {result['Crack_Detected']}"
        )
        print(
            f"Crack Regions  : {result['Crack_Regions']}"
        )

        if CALIBRATION_ENABLED:

            print(
                f"Maximum Width  : "
                f"{result['Maximum_Width_MM']:.3f} mm"
            )

            print(
                f"Average Width  : "
                f"{result['Average_Width_MM']:.3f} mm"
            )

            print(
                f"Crack Length   : "
                f"{result['Crack_Length_MM']:.3f} mm"
            )

            print(
                f"Crack Area     : "
                f"{result['Crack_Area_MM2']:.3f} mm²"
            )

        else:

            print(
                f"Maximum Width  : "
                f"{result['Maximum_Width_Pixels']:.3f} pixels"
            )

            print(
                f"Average Width  : "
                f"{result['Average_Width_Pixels']:.3f} pixels"
            )

            print(
                f"Crack Length   : "
                f"{result['Crack_Length_Pixels']:.3f} pixels"
            )

            print(
                f"Crack Area     : "
                f"{result['Crack_Area_Pixels']:.3f} pixels²"
            )

        print(
            f"Severity       : {result['Severity']}"
        )

        print(
            f"Risk Level     : {result['Risk_Level']}"
        )

        print(
            f"Message        : {result['Automatic_Message']}"
        )

print("[✓ SUCCESS] Final results displayed successfully.")

# SECTION 31: DISPLAY COMPLETE PROCESSING PIPELINE

In [ ]:
# WORKING: Display complete image-processing stages

if SHOW_PIPELINE:

    for filename, stages in all_stages.items():

        print()
        print("===============================================================")
        print(f"PIPELINE: {filename}")
        print("===============================================================")

        display_pipeline(
            stages,
            filename
        )

print("[✓ SUCCESS] Complete processing pipeline displayed successfully.")

# SECTION 32: FINAL PROJECT STATISTICS

In [ ]:
# WORKING: Calculate overall project statistics

total_uploaded = len(
    image_paths
)

total_successful = len(
    successful_files
)

total_failed = len(
    failed_files
)

if not summary_df.empty:

    crack_detected_count = int(
        (
            summary_df["Crack_Detected"]
            == "YES"
        ).sum()
    )

    total_crack_regions = int(
        summary_df["Crack_Regions"]
        .sum()
    )

    good_count = int(
        (
            summary_df["Severity"]
            == "GOOD"
        ).sum()
    )

    medium_count = int(
        (
            summary_df["Severity"]
            == "MEDIUM"
        ).sum()
    )

    heavy_count = int(
        (
            summary_df["Severity"]
            == "HEAVY"
        ).sum()
    )

    if CALIBRATION_ENABLED:

        valid_max_widths = pd.to_numeric(
            summary_df[
                "Maximum_Width_MM"
            ],
            errors="coerce"
        ).dropna()

        if len(valid_max_widths) > 0:

            overall_max_width = float(
                valid_max_widths.max()
            )

            overall_average_width = float(
                valid_max_widths.mean()
            )

        else:

            overall_max_width = np.nan
            overall_average_width = np.nan

    else:

        valid_max_widths = pd.to_numeric(
            summary_df[
                "Maximum_Width_Pixels"
            ],
            errors="coerce"
        ).dropna()

        if len(valid_max_widths) > 0:

            overall_max_width = float(
                valid_max_widths.max()
            )

            overall_average_width = float(
                valid_max_widths.mean()
            )

        else:

            overall_max_width = np.nan
            overall_average_width = np.nan

else:

    crack_detected_count = 0
    total_crack_regions = 0
    good_count = 0
    medium_count = 0
    heavy_count = 0
    overall_max_width = np.nan
    overall_average_width = np.nan

print()
print("PROJECT STATISTICS")
print("---------------------------------------------------------------")
print(f"Total Images Uploaded      : {total_uploaded}")
print(f"Successfully Processed     : {total_successful}")
print(f"Failed                     : {total_failed}")
print(f"Images With Cracks         : {crack_detected_count}")
print(f"Total Crack Regions        : {total_crack_regions}")
print(f"GOOD                       : {good_count}")
print(f"MEDIUM                     : {medium_count}")
print(f"HEAVY RISK                 : {heavy_count}")

if CALIBRATION_ENABLED:

    if np.isfinite(overall_max_width):

        print(
            f"Overall Maximum Width      : "
            f"{overall_max_width:.3f} mm"
        )

        print(
            f"Average Image Max Width   : "
            f"{overall_average_width:.3f} mm"
        )

    else:

        print(
            "Overall Maximum Width      : N/A"
        )

else:

    if np.isfinite(overall_max_width):

        print(
            f"Overall Maximum Width      : "
            f"{overall_max_width:.3f} pixels"
        )

        print(
            f"Average Image Max Width   : "
            f"{overall_average_width:.3f} pixels"
        )

    else:

        print(
            "Overall Maximum Width      : N/A"
        )

print("Project statistics generated successfully.")

# SECTION 33: ERROR REPORT

In [ ]:
# WORKING: Save failed-image information

error_report_path = os.path.join(
    REPORT_DIR,
    "processing_errors.csv"
)

if len(failed_files) > 0:

    error_df = pd.DataFrame(
        failed_files
    )

else:

    error_df = pd.DataFrame(
        columns=[
            "filename",
            "error"
        ]
    )

error_df.to_csv(
    error_report_path,
    index=False
)

if len(failed_files) == 0:

    print(
        "No processing errors occurred."
    )

else:

    print(
        f"Number of failed images: "
        f"{len(failed_files)}"
    )

    display(
        error_df
    )

print(
    f"Error report saved: "
    f"{error_report_path}"
)

print("Error handling/reporting completed.")

# SECTION 34: CREATE ZIP RESULT PACKAGE

In [ ]:
# WORKING: Create complete ZIP package containing all results

ZIP_PATH = "crack_results.zip"

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for root, dirs, filenames in os.walk(
        OUTPUT_DIR
    ):

        for filename in filenames:

            file_path = os.path.join(
                root,
                filename
            )

            archive_name = os.path.relpath(
                file_path,
                os.path.dirname(
                    OUTPUT_DIR
                )
            )

            zip_file.write(
                file_path,
                archive_name
            )

zip_size_mb = (
    os.path.getsize(
        ZIP_PATH
    ) / (1024 * 1024)
)

print(
    f"ZIP created: {ZIP_PATH}"
)

print(
    f"ZIP size: {zip_size_mb:.2f} MB"
)

print("Complete result ZIP created successfully.")

#  SECTION 35: VERIFY OUTPUT FILES

In [ ]:
# WORKING: Verify important project output files

required_paths = [
    OUTPUT_DIR,
    ANNOTATED_DIR,
    MASK_DIR,
    PROCESSED_DIR,
    REPORT_DIR,
    research_csv_path,
    measurement_csv_path,
    error_report_path,
    ZIP_PATH
]

all_outputs_valid = True

for path in required_paths:

    exists = os.path.exists(
        path
    )

    if exists:

        print(
            f"{path}"
        )

    else:

        all_outputs_valid = False

        print(
            f" {path}"
        )

if all_outputs_valid:

    print(
        "All required output files/directories verified."
    )

else:

    print(
        "Some output files/directories are missing."
    )

# SECTION 36: DOWNLOAD COMPLETE ZIP RESULT

In [ ]:
# WORKING: Download complete research result package

if os.path.exists(ZIP_PATH):

    print(
        f"Ready for download: {ZIP_PATH}"
    )

    files.download(
        ZIP_PATH
    )

    print(
        "ZIP download initiated successfully."
    )

else:

    print(
        "ZIP file was not found."
    )

# SECTION 37: FINAL PROJECT SUMMARY

In [ ]:
print(
    "Project:"
)
print(
    "OpenCV-Based System for Crack Identification and Width Estimation"
)

print()
print(
    "---------------------------------------------------------------"
)

print(
    f"Total Images Uploaded       : {total_uploaded}"
)

print(
    f"Successfully Processed      : {total_successful}"
)

print(
    f"Failed Images               : {total_failed}"
)

print(
    f"Images With Crack           : {crack_detected_count}"
)

print(
    f"Total Crack Regions         : {total_crack_regions}"
)

print(
    f"GOOD / LOW RISK             : {good_count}"
)

print(
    f"MEDIUM / MEDIUM RISK        : {medium_count}"
)

print(
    f"HEAVY / HIGH RISK           : {heavy_count}"
)

print()

if CALIBRATION_ENABLED:

    print(
        "Calibration                 : ENABLED"
    )

    print(
        f"Pixels Per MM               : {PIXELS_PER_MM}"
    )

    if np.isfinite(overall_max_width):

        print(
            f"Maximum Detected Width     : "
            f"{overall_max_width:.3f} mm"
        )

        print(
            f"Average Maximum Width      : "
            f"{overall_average_width:.3f} mm"
        )

else:

    print(
        "Calibration                 : DISABLED"
    )

    if np.isfinite(overall_max_width):

        print(
            f"Maximum Detected Width     : "
            f"{overall_max_width:.3f} pixels"
        )

        print(
            f"Average Maximum Width      : "
            f"{overall_average_width:.3f} pixels"
        )

print()
print(
    "---------------------------------------------------------------"
)

print(
    f"Output Directory            : {OUTPUT_DIR}/"
)

print(
    f"Annotated Images            : {ANNOTATED_DIR}/"
)

print(
    f"Crack Masks                 : {MASK_DIR}/"
)

print(
    f"Processed Images            : {PROCESSED_DIR}/"
)

print(
    f"Research CSV                : {research_csv_path}"
)

print(
    f"Measurement CSV             : {measurement_csv_path}"
)

print(
    f"Error Report                : {error_report_path}"
)

print(
    f"ZIP Result                  : {ZIP_PATH}"
)

print()
print(
    "CORE WIDTH FORMULA:"
)

print(
    "Width = 2 × Distance Transform Value"
)

if CALIBRATION_ENABLED:

    print(
        "Width(mm) = "
        "(2 × Distance Transform Value) / Pixels_Per_MM"
    )

print(
    "IMPORTANT:"
)

print(
    "This system is intended for image-based crack detection and "
    "measurement research. Crack severity thresholds are configurable "
    "general guidelines and should be validated against relevant "
    "engineering standards, inspection procedures, and application "
    "requirements."
)


# SECTION 38: QUICK RESULT TABLE

In [ ]:
# WORKING: Create a compact result table for presentation/demo

if not final_report_df.empty:

    quick_columns = [
        "Image_Name",
        "Crack_Detected",
        "Crack_Regions",
        "Maximum_Width_MM",
        "Average_Width_MM",
        "Crack_Length_MM",
        "Severity",
        "Risk_Level",
        "Health_Status"
    ]

    if not CALIBRATION_ENABLED:

        quick_columns = [
            "Image_Name",
            "Crack_Detected",
            "Crack_Regions",
            "Maximum_Width_Pixels",
            "Average_Width_Pixels",
            "Crack_Length_Pixels",
            "Severity",
            "Risk_Level",
            "Health_Status"
        ]

    quick_columns = [
        c
        for c in quick_columns
        if c in final_report_df.columns
    ]

    quick_df = final_report_df[
        quick_columns
    ].copy()

    display(
        quick_df.round(3)
    )

else:

    print("No results available.")

print("Quick result table generated successfully.")

# SECTION 39: FORMULA VERIFICATION EXAMPLE

In [ ]:
# WORKING: Verify the research width-estimation formula

example_distance = 3.5

example_width_pixels = (
    2 * example_distance
)

print(
    f"Distance Transform Value = "
    f"{example_distance} pixels"
)

print(
    f"Width = 2 × {example_distance}"
)

print(
    f"Width = {example_width_pixels} pixels"
)

if CALIBRATION_ENABLED:

    example_width_mm = (
        example_width_pixels /
        PIXELS_PER_MM
    )

    print(
        f"Width = {example_width_pixels} / "
        f"{PIXELS_PER_MM}"
    )

    print(
        f"Actual Width = "
        f"{example_width_mm:.3f} mm"
    )

print()
print(
    " Width formula verification completed."
)